In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
DEBUG = False        # True = test on 200 clips first. Set False for the real run.
NUM_FRAMES = 16     # frames sampled per clip
IMG_SIZE = 224
SEED = 42
BATCH_CLIPS = 16    # clips per batch (16 clips x 16 frames = 256 images)

In [6]:
import glob, os, shutil, subprocess

RAR_DIR = glob.glob('/kaggle/input/**/cricshot10k_raw', recursive=True)[0]
rars = sorted(glob.glob(os.path.join(RAR_DIR, '*.rar')))
print('RAR folder:', RAR_DIR, '| archives found:', len(rars))

# Make sure an extractor is available
if shutil.which('unrar') is None and shutil.which('7z') is None:
    subprocess.run('apt-get -qq update && apt-get -qq install -y unrar p7zip-full', shell=True)
print('unrar:', shutil.which('unrar'), '| 7z:', shutil.which('7z'))

os.makedirs('/tmp/videos', exist_ok=True)
for r in rars:
    if shutil.which('unrar'):
        cmd = ['unrar', 'x', '-o+', '-idq', r, '/tmp/videos/']
    else:
        cmd = ['7z', 'x', '-y', '-bso0', '-bsp0', r, '-o/tmp/videos']
    res = subprocess.run(cmd, capture_output=True, text=True)
    status = 'OK' if res.returncode == 0 else f'FAILED ({res.returncode}): {res.stderr[-300:]}'
    print(f'{os.path.basename(r):25s} {status}')

!du -sh /tmp/videos
!find /tmp/videos -maxdepth 2 -type d | head -20

RAR folder: /kaggle/input/notebooks/srimanreddybommi/01-download-cricshot10k/cricshot10k_raw | archives found: 15
unrar: /usr/bin/unrar | 7z: /usr/bin/7z
Cover Drive.rar           OK
Defensive.rar             OK
Down The Wicket.rar       OK
Flick.rar                 OK
Hook.rar                  OK
Late Cut.rar              OK
Lofted Legside.rar        OK
Lofted Offside.rar        OK
Pull.rar                  OK
Reverse Sweep.rar         OK
Scoop.rar                 OK
Square Cut.rar            OK
Straight Drive.rar        OK
Sweep.rar                 OK
Upper Cut.rar             OK
3.7G	/tmp/videos
/tmp/videos
/tmp/videos/Cover Drive
/tmp/videos/Straight Drive
/tmp/videos/Lofted Offside
/tmp/videos/Lofted Legside
/tmp/videos/Defensive
/tmp/videos/Flick
/tmp/videos/Upper Cut
/tmp/videos/Pull
/tmp/videos/Reverse Sweep
/tmp/videos/Square Cut
/tmp/videos/Scoop
/tmp/videos/Sweep
/tmp/videos/Late Cut
/tmp/videos/Down The Wicket
/tmp/videos/Hook


In [7]:
import re, numpy as np, pandas as pd, cv2

ROOT = '/tmp/videos'
rows = []
for dirpath, _, files in os.walk(ROOT):
    for f in sorted(files):
        m = re.match(r'vid(\d+)_(\d+)\.avi$', f, re.IGNORECASE)
        if m is None:
            continue
        rows.append({'path': os.path.relpath(os.path.join(dirpath, f), ROOT),
                     'class': os.path.basename(dirpath),
                     'match_id': int(m.group(1)), 'clip_id': int(m.group(2))})

assert len(rows) > 0, "No videos found in /tmp/videos - re-run the extraction cell (Cell 2)"

df = pd.DataFrame(rows)
classes = sorted(df['class'].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
df['label'] = df['class'].map(class_to_idx)

print('Clips:', len(df), '| Classes:', len(classes), '| Matches:', df['match_id'].nunique())
print(df['class'].value_counts().sort_index())
print(df.head())

Clips: 10091 | Classes: 15 | Matches: 534
class
Cover Drive         772
Defensive           605
Down The Wicket     890
Flick               933
Hook                534
Late Cut            306
Lofted Legside     1123
Lofted Offside     1023
Pull                745
Reverse Sweep       252
Scoop               279
Square Cut         1023
Straight Drive      423
Sweep               908
Upper Cut           275
Name: count, dtype: int64
                        path        class  match_id  clip_id  label
0  Cover Drive/vid100_41.avi  Cover Drive       100       41      0
1  Cover Drive/vid100_60.avi  Cover Drive       100       60      0
2   Cover Drive/vid100_8.avi  Cover Drive       100        8      0
3  Cover Drive/vid101_24.avi  Cover Drive       101       24      0
4  Cover Drive/vid101_39.avi  Cover Drive       101       39      0


In [8]:
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

# Match-level split: no match appears in more than one set (~70/15/15)
df['split_match'] = 'train'
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
rest_idx, test_idx = next(sgkf.split(df, df['label'], groups=df['match_id']))
df.loc[df.index[test_idx], 'split_match'] = 'test'

rest = df.iloc[rest_idx]
sgkf2 = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
_, val_idx = next(sgkf2.split(rest, rest['label'], groups=rest['match_id']))
df.loc[rest.index[val_idx], 'split_match'] = 'val'

# Random split (the "leaky" way earlier papers did it), same proportions, for comparison
tr, te = train_test_split(df.index, test_size=0.15, stratify=df['label'], random_state=SEED)
tr, va = train_test_split(tr, test_size=0.15/0.85, stratify=df.loc[tr, 'label'], random_state=SEED)
df['split_random'] = 'train'
df.loc[va, 'split_random'] = 'val'
df.loc[te, 'split_random'] = 'test'

# Checks
for s in ['train', 'val', 'test']:
    sub = df[df.split_match == s]
    print(f"{s:5s}: {len(sub):5d} clips, {sub.match_id.nunique():3d} matches")

m = {s: set(df[df.split_match == s].match_id) for s in ['train', 'val', 'test']}
assert not (m['train'] & m['val']) and not (m['train'] & m['test']) and not (m['val'] & m['test'])
print('No match overlap between splits ✅')

print(pd.crosstab(df['class'], df['split_match']))

train:  7254 clips, 380 matches
val  :  1487 clips,  78 matches
test :  1350 clips,  76 matches
No match overlap between splits ✅
split_match      test  train  val
class                            
Cover Drive       105    556  111
Defensive          81    443   81
Down The Wicket   114    650  126
Flick             120    696  117
Hook               87    373   74
Late Cut           29    223   54
Lofted Legside    171    785  167
Lofted Offside    144    740  139
Pull               82    522  141
Reverse Sweep      31    191   30
Scoop              34    189   56
Square Cut        137    749  137
Straight Drive     57    298   68
Sweep             115    642  151
Upper Cut          43    197   35


In [9]:
import torch
from torch.utils.data import Dataset, DataLoader
cv2.setNumThreads(0)

class ClipDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        cap = cv2.VideoCapture(os.path.join(ROOT, self.paths[i]))
        frames = []
        while True:
            ok, f = cap.read()
            if not ok:
                break
            frames.append(f)
        cap.release()
        n = len(frames)
        if n == 0:   # unreadable video
            return torch.zeros(NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3, dtype=torch.uint8), 0, i
        idx = np.linspace(0, n - 1, NUM_FRAMES).round().astype(int)   # uniform sampling
        clip = np.stack([cv2.resize(cv2.cvtColor(frames[j], cv2.COLOR_BGR2RGB),
                                    (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
                         for j in idx])
        return torch.from_numpy(clip), n, i     # (T, H, W, 3) uint8

work_df = df.sample(200, random_state=SEED).reset_index(drop=True) if DEBUG else df
loader = DataLoader(ClipDataset(work_df['path'].tolist()), batch_size=BATCH_CLIPS,
                    shuffle=False, num_workers=4, pin_memory=True)
print('Clips to process:', len(work_df))

Clips to process: 200


In [10]:
import torch.nn as nn, torchvision
from torchvision.models import EfficientNet_B0_Weights

device = 'cuda'
m = torchvision.models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
backbone = nn.Sequential(m.features, m.avgpool, nn.Flatten()).to(device).eval()  # outputs 1280 per image

MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

@torch.no_grad()
def embed(x):                       # x: (B, T, H, W, 3) uint8 on GPU
    B, T = x.shape[:2]
    x = x.permute(0, 1, 4, 2, 3).reshape(B * T, 3, IMG_SIZE, IMG_SIZE).float() / 255.
    x = (x - MEAN) / STD
    with torch.autocast('cuda', dtype=torch.float16):
        f = backbone(x)
    return f.view(B, T, -1).float().cpu().numpy()

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 176MB/s]


In [11]:
from tqdm.auto import tqdm
import time

N = len(work_df)
feats      = np.zeros((N, NUM_FRAMES, 1280), dtype=np.float16)
feats_flip = np.zeros((N, NUM_FRAMES, 1280), dtype=np.float16)
n_frames   = np.zeros(N, dtype=np.int32)

t0 = time.time()
for clips, n, idx in tqdm(loader):
    clips = clips.to(device, non_blocking=True)
    feats[idx.numpy()]      = embed(clips)
    feats_flip[idx.numpy()] = embed(torch.flip(clips, dims=[3]))   # flip left-right (augmentation)
    n_frames[idx.numpy()]   = n.numpy()

print(f'Done in {(time.time() - t0) / 60:.1f} min')
print('Unreadable clips:', int((n_frames == 0).sum()))
print('Feature shape:', feats.shape, '| any NaN:', np.isnan(feats.astype(np.float32)).any())

  0%|          | 0/13 [00:00<?, ?it/s]

Done in 0.5 min
Unreadable clips: 0
Feature shape: (200, 16, 1280) | any NaN: False


In [12]:
import json
OUT = '/kaggle/working'
work_df['n_frames'] = n_frames
work_df.to_csv(f'{OUT}/metadata.csv', index=False)
np.save(f'{OUT}/feats_b0.npy', feats)
np.save(f'{OUT}/feats_b0_flip.npy', feats_flip)
json.dump({'classes': classes, 'num_frames': NUM_FRAMES, 'img_size': IMG_SIZE,
           'backbone': 'efficientnet_b0_imagenet1k_v1', 'seed': SEED},
          open(f'{OUT}/config.json', 'w'), indent=2)
!ls -lh /kaggle/working

total 16M
-rw-r--r-- 1 root root  381 Oct 25 21:54 config.json
-rw-r--r-- 1 root root 7.9M Oct 25 21:54 feats_b0_flip.npy
-rw-r--r-- 1 root root 7.9M Oct 25 21:54 feats_b0.npy
-rw-r--r-- 1 root root  12K Oct 25 21:54 metadata.csv
